# 참고용 사본 (Reference copy)

이 노트북은 원 연구 노트북에서 출력, 실행 횟수, 로컬 경로, 키 값을 제거한 **참고용 사본**이다. 뉴스 원문, 일별·분봉 가격, 경매 거래대금 등 **공개하지 않은 데이터**(`news_tokenized_labeled*.pkl`, `daily_adjusted_close.parquet` 등)가 있어야 처음부터 끝까지 실행되며, 이 저장소만으로는 실행되지 않는다. 분석 구조와 코드를 읽는 용도다. 이 저장소에서 바로 실행되는 것은 `00_public_sample_workflow.ipynb`와 `../code/`뿐이다.

This is a reference copy of an original research notebook: outputs, execution counts, local paths and keys were removed. It needs non-public data (news text, daily/minute prices, auction values) and will not run from this repository alone.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings(
    "ignore",
    message="Downcasting behavior in `replace` is deprecated",
    category=FutureWarning,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.6f}".format)


---
### **CTO Portfolio Capacity and Cost**


In [ ]:
# =============================================================================
# 1. 설정
# =============================================================================
MODEL = "NB"              # "NB" or "SESTM"
SIGNAL_RULE = "topn"      # "topn" or "threshold"
UNIVERSE = "kospi200"     # "all" or "kospi200"
TOP_N = 10
THRESHOLD = 0.9

START_DATE = "2020"
PARTICIPATION_RATE = 0.05
AUM = 10_000_000_000      # 100억원. None이면 AUM 기반 체결수익률 계산 생략
REALLOCATE_UNFILLED = False

# 비용/슬리피지 시나리오. bp 단위.
BUY_FEE_BP = 1.0
SELL_FEE_BP = 1.0
SELL_TAX_BP = 15.0
BUY_SLIPPAGE_BP = 0.0
SELL_SLIPPAGE_BP = 0.0

# return source:
# "auction": 수정종가 -> 익일 수정시가
# "vwap": 종가 윈도우 VWAP -> 익일 09:00~09:05 VWAP, 수정가격 스케일에 맞게 보정
RETURN_SOURCE = "auction"


In [ ]:
# =============================================================================
# 2. 데이터 불러오기
# =============================================================================
SESTM_sentiment_wide = pd.read_parquet("SESTM_sentiment_wide.parquet")
NB_sentiment_wide = pd.read_parquet("NB_sentiment_wide.parquet")
price_wide = pd.read_parquet("price_wide.parquet")
open_wide = pd.read_parquet("adjusted_open_wide.parquet").reindex_like(price_wide)
df = pd.read_pickle("news_tokenized_labeled_scored.pkl")
kospi200_member_wide = pd.read_parquet("kospi200_member_wide.parquet").reindex_like(price_wide)

auction = pd.read_parquet("auction_liquidity_daily.parquet")
auction = auction[auction["gicode"].isin(price_wide.columns)].copy()

print(price_wide.shape, open_wide.shape, auction.shape)
auction.head()


In [ ]:
# =============================================================================
# 2-1. Universe 확인
# =============================================================================
universe_mask = kospi200_member_wide.notna() if UNIVERSE == "kospi200" else price_wide.notna()

display(
    pd.Series({
        "Universe": UNIVERSE,
        "Mean Universe Names": universe_mask.sum(axis=1).mean(),
        "Median Universe Names": universe_mask.sum(axis=1).median(),
        "Min Universe Names": universe_mask.sum(axis=1).min(),
        "Max Universe Names": universe_mask.sum(axis=1).max(),
    })
)


In [ ]:
# =============================================================================
# 3. 장중 뉴스 기반 sentiment wide 재생성
# =============================================================================
source_date = pd.to_datetime(df["source_date"].astype(str), format="%Y%m%d", errors="coerce")
timestamp_date = pd.to_datetime(
    df["timestamp_date"].astype("string").str.strip().replace("", pd.NA),
    errors="coerce",
)
timestamp_time = pd.to_timedelta(
    df["timestamp_time"].astype("string").str.strip().replace("", pd.NA),
    errors="coerce",
)

intraday_mask = (
    timestamp_date.notna()
    & timestamp_time.notna()
    & timestamp_date.eq(source_date)
    & timestamp_time.between(pd.Timedelta(hours=9), pd.Timedelta(hours=15))
)

df_intraday = df.loc[intraday_mask].copy()

tmp = df_intraday[["source_date", "gicode", "SESTM_sentiment", "NB_sentiment"]].copy()
tmp["date"] = pd.to_datetime(
    tmp["source_date"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True),
    format="%Y%m%d",
    errors="coerce",
).dt.strftime("%Y-%m-%d")
tmp["gicode"] = tmp["gicode"].astype("string").str.strip()
tmp = tmp.dropna(subset=["date", "gicode"])
tmp = tmp[tmp["date"].isin(price_wide.index) & tmp["gicode"].isin(price_wide.columns)]

SESTM_intraday_wide = (
    tmp.pivot_table(index="date", columns="gicode", values="SESTM_sentiment", aggfunc="mean")
       .reindex(index=price_wide.index, columns=price_wide.columns)
       .astype("float32")
)

NB_intraday_wide = (
    tmp.pivot_table(index="date", columns="gicode", values="NB_sentiment", aggfunc="mean")
       .reindex(index=price_wide.index, columns=price_wide.columns)
       .astype("float32")
)

print(df_intraday.shape)
NB_intraday_wide.notna().sum(axis=1).describe()


In [ ]:
# =============================================================================
# 4. auction liquidity wide 구성
# =============================================================================
def pivot_auction(col):
    return (
        auction.pivot(index="date", columns="gicode", values=col)
        .reindex(index=price_wide.index, columns=price_wide.columns)
        .astype("float64")
    )

open_value_auction = pivot_auction("open_value_auction")
close_value_auction = pivot_auction("close_value_auction")
open_value_window = pivot_auction("open_value_0900_0905")
close_value_window = pivot_auction("close_value_window")

open_price_auction_raw = pivot_auction("open_price_auction")
close_price_auction_raw = pivot_auction("close_price_auction")
open_vwap_raw = pivot_auction("open_vwap_0900_0905")
close_vwap_raw = pivot_auction("close_vwap_window")

# 기본 capacity: day0 종가 단일가 매수, day1 시가 단일가 매도 가능 거래대금의 작은 쪽
capacity_auction_signal_day = (
    np.minimum(close_value_auction, open_value_auction.shift(-1)) * PARTICIPATION_RATE
)

# VWAP robustness용 capacity: 종가 윈도우, 익일 시가 윈도우 거래대금 기준
capacity_vwap_signal_day = (
    np.minimum(close_value_window, open_value_window.shift(-1)) * PARTICIPATION_RATE
)

display(capacity_auction_signal_day.stack().describe(percentiles=[0.25, 0.5, 0.75, 0.95]))


In [ ]:
# =============================================================================
# 5. 수익률 데이터
# =============================================================================
CTO_auction = open_wide / price_wide.shift(1) - 1

# raw VWAP를 수정가격 스케일에 맞춰 보정합니다.
# 분봉 가격 자체는 raw이므로, 당일 raw auction price와 daily adjusted price의 비율을 보정계수로 사용합니다.
close_adj_factor = price_wide / close_price_auction_raw
open_adj_factor = open_wide / open_price_auction_raw

close_vwap_adj = close_vwap_raw * close_adj_factor
open_vwap_adj = open_vwap_raw * open_adj_factor

CTO_vwap = open_vwap_adj / close_vwap_adj.shift(1) - 1

def apply_roundtrip_cost(
    gross_ret,
    buy_fee_bp=BUY_FEE_BP,
    sell_fee_bp=SELL_FEE_BP,
    sell_tax_bp=SELL_TAX_BP,
    buy_slippage_bp=BUY_SLIPPAGE_BP,
    sell_slippage_bp=SELL_SLIPPAGE_BP,
):
    buy_cost = (buy_fee_bp + buy_slippage_bp) / 10_000
    sell_cost = (sell_fee_bp + sell_tax_bp + sell_slippage_bp) / 10_000
    return (1 + gross_ret) * (1 - sell_cost) / (1 + buy_cost) - 1

gross_ret_wide = CTO_vwap if RETURN_SOURCE == "vwap" else CTO_auction
net_ret_wide = apply_roundtrip_cost(gross_ret_wide)

gross_ret_wide.loc[START_DATE:].stack().describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])


---
### **Strategy Functions**


In [ ]:
def make_signal(sentiment_wide, rule=SIGNAL_RULE, top_n=TOP_N, threshold=THRESHOLD):
    sentiment_wide = sentiment_wide.where(universe_mask)
    if rule == "topn":
        return (
            (sentiment_wide.rank(axis=1, method="first", ascending=False) <= top_n)
            .where(sentiment_wide.notna())
            .replace(False, np.nan)
            .astype(float)
        )
    if rule == "threshold":
        return (
            (sentiment_wide >= threshold)
            .where(sentiment_wide.notna())
            .replace(False, np.nan)
            .astype(float)
        )
    raise ValueError("rule must be 'topn' or 'threshold'")


def performance_stats(r, periods_per_year=252):
    r = r.dropna().astype(float)
    if r.empty:
        return pd.Series(dtype=float)
    cum = (1 + r).cumprod()
    years = len(r) / periods_per_year
    return pd.Series({
        "CAGR": cum.iloc[-1] ** (1 / years) - 1 if years > 0 else np.nan,
        "Annualized Return": r.mean() * periods_per_year,
        "Annualized Vol": r.std(ddof=1) * np.sqrt(periods_per_year),
        "Sharpe": r.mean() / r.std(ddof=1) * np.sqrt(periods_per_year) if r.std(ddof=1) > 0 else np.nan,
        "MDD": (cum / cum.cummax() - 1).min(),
        "Win Rate": (r > 0).mean(),
        "N Days": len(r),
    })


def backtest_with_capacity(
    signal_day,
    gross_ret,
    net_ret,
    capacity_signal_day,
    *,
    aum=AUM,
    start_date=START_DATE,
):
    # signal_day는 day0 종가 매수 신호입니다.
    # CTO 수익률은 day1 index에 있으므로 signal을 1거래일 shift합니다.
    positions_exit_day = signal_day.shift(1)

    gross_equal_ret = (positions_exit_day * gross_ret).mean(axis=1)
    net_equal_ret = (positions_exit_day * net_ret).mean(axis=1)

    selected_count_signal_day = signal_day.notna().sum(axis=1)
    cap_selected_signal_day = capacity_signal_day.where(signal_day.notna())
    cap_positive_signal_day = capacity_signal_day.where(signal_day.notna() & capacity_signal_day.gt(0))
    tradable_count_signal_day = cap_positive_signal_day.notna().sum(axis=1)

    total_capacity_exit_day = cap_positive_signal_day.sum(axis=1).shift(1)
    min_capacity_exit_day = cap_positive_signal_day.min(axis=1).shift(1)
    p10_capacity_exit_day = cap_positive_signal_day.quantile(0.10, axis=1).shift(1)
    median_capacity_exit_day = cap_positive_signal_day.median(axis=1).shift(1)
    equal_weight_full_fill_aum_capacity = (
        cap_positive_signal_day.min(axis=1) * selected_count_signal_day
    ).shift(1)

    result = pd.DataFrame({
        "gross_equal_ret": gross_equal_ret,
        "net_equal_ret": net_equal_ret,
        "n_names": selected_count_signal_day.shift(1),
        "n_tradable_names": tradable_count_signal_day.shift(1),
        "total_capacity": total_capacity_exit_day,
        "min_name_capacity": min_capacity_exit_day,
        "p10_name_capacity": p10_capacity_exit_day,
        "median_name_capacity": median_capacity_exit_day,
        "equal_weight_full_fill_aum_capacity": equal_weight_full_fill_aum_capacity,
    })

    if aum is not None:
        denominator = (
            tradable_count_signal_day if REALLOCATE_UNFILLED else selected_count_signal_day
        ).replace(0, np.nan)
        target_per_name = aum / denominator
        fill_signal_day = cap_positive_signal_day.clip(upper=target_per_name, axis=0)
        fill_exit_day = fill_signal_day.shift(1)
        result["invested"] = fill_exit_day.sum(axis=1)
        result["utilization"] = result["invested"] / aum
        result["gross_ret_on_aum"] = (fill_exit_day * gross_ret).sum(axis=1) / aum
        result["net_ret_on_aum"] = (fill_exit_day * net_ret).sum(axis=1) / aum
        result["net_ret_on_invested"] = (
            (fill_exit_day * net_ret).sum(axis=1) / result["invested"].replace(0, np.nan)
        )

    result = result.loc[start_date:].copy()
    return result


def report_backtest(bt, label):
    constrained_col = "net_ret_on_aum" if "net_ret_on_aum" in bt.columns else "net_equal_ret"
    perf = performance_stats(bt[constrained_col]).add_prefix("AUM_")
    unconstrained = performance_stats(bt["net_equal_ret"]).add_prefix("EqualWeight_")
    capacity = pd.Series({
        "Selected Names Avg": bt["n_names"].mean(),
        "Tradable Names Avg": bt["n_tradable_names"].mean(),
        "No Position/Tradable Days": (bt["n_tradable_names"].fillna(0) == 0).sum(),
        "Median Total Capacity (억원)": bt["total_capacity"].replace(0, np.nan).median() / 100_000_000,
        "P10 Name Capacity (억원)": bt["p10_name_capacity"].median() / 100_000_000,
        "Median Name Capacity (억원)": bt["median_name_capacity"].median() / 100_000_000,
        "Equal-Weight Full-Fill AUM Cap (억원)": bt["equal_weight_full_fill_aum_capacity"].median() / 100_000_000,
    })
    if "invested" in bt.columns:
        capacity["Median Invested (억원)"] = bt["invested"].median() / 100_000_000
        capacity["Mean Utilization"] = bt["utilization"].mean()
        capacity["Median Utilization"] = bt["utilization"].median()
        capacity["Invested-Day Ret Mean"] = bt["net_ret_on_invested"].mean()
    return pd.concat([pd.Series({"Strategy": label}), perf, unconstrained, capacity])


---
### **Example: 현재 설정으로 실행**

성과표에는 두 종류의 수익률이 함께 표시됩니다.

- `AUM_*`: 분봉 거래대금 capacity로 실제 투자 가능 금액을 제한한 수익률입니다.
- `EqualWeight_*`: 모든 선택 종목을 동일가중으로 전부 체결할 수 있다고 보는 비현실적 기준입니다.

운용규모 판단에는 `AUM_*`, `Median Invested`, `Utilization`, `Tradable Names`를 중심으로 봅니다.


In [ ]:
sentiment = NB_intraday_wide if MODEL == "NB" else SESTM_intraday_wide
signal = make_signal(sentiment, rule=SIGNAL_RULE, top_n=TOP_N, threshold=THRESHOLD)

capacity_signal_day = (
    capacity_vwap_signal_day if RETURN_SOURCE == "vwap" else capacity_auction_signal_day
)

bt = backtest_with_capacity(
    signal,
    gross_ret_wide,
    net_ret_wide,
    capacity_signal_day,
    aum=AUM,
    start_date=START_DATE,
)

label = f"{MODEL}_{UNIVERSE}_{SIGNAL_RULE}_{TOP_N if SIGNAL_RULE == 'topn' else THRESHOLD}_{RETURN_SOURCE}"
display(report_backtest(bt, label))
bt.tail()


---
### **여러 설정 비교**


In [ ]:
tests = []

for n in [1, 3, 5, 10]:
    sig = make_signal(NB_intraday_wide, rule="topn", top_n=n)
    b = backtest_with_capacity(sig, gross_ret_wide, net_ret_wide, capacity_signal_day, aum=AUM)
    tests.append(report_backtest(b, f"NB_{UNIVERSE}_top{n}_{RETURN_SOURCE}"))

for th in [0.7, 0.8, 0.9, 0.95]:
    sig = make_signal(NB_intraday_wide, rule="threshold", threshold=th)
    b = backtest_with_capacity(sig, gross_ret_wide, net_ret_wide, capacity_signal_day, aum=AUM)
    tests.append(report_backtest(b, f"NB_{UNIVERSE}_th{th}_{RETURN_SOURCE}"))

comparison = pd.DataFrame(tests).set_index("Strategy")
comparison


---
### **Auction vs VWAP 강건성**


In [ ]:
sig = make_signal(NB_intraday_wide, rule=SIGNAL_RULE, top_n=TOP_N, threshold=THRESHOLD)

bt_auction = backtest_with_capacity(
    sig,
    CTO_auction,
    apply_roundtrip_cost(CTO_auction),
    capacity_auction_signal_day,
    aum=AUM,
)

bt_vwap = backtest_with_capacity(
    sig,
    CTO_vwap,
    apply_roundtrip_cost(CTO_vwap),
    capacity_vwap_signal_day,
    aum=AUM,
)

pd.DataFrame([
    report_backtest(bt_auction, "auction_price"),
    report_backtest(bt_vwap, "vwap_window"),
]).set_index("Strategy")


---
### **비용/슬리피지 민감도**


In [ ]:
cost_tests = []
sig = make_signal(NB_intraday_wide, rule=SIGNAL_RULE, top_n=TOP_N, threshold=THRESHOLD)

for slippage_bp in [0, 2.5, 5, 10, 20]:
    net_ret_scenario = apply_roundtrip_cost(
        gross_ret_wide,
        buy_slippage_bp=slippage_bp,
        sell_slippage_bp=slippage_bp,
    )
    b = backtest_with_capacity(
        sig,
        gross_ret_wide,
        net_ret_scenario,
        capacity_signal_day,
        aum=AUM,
    )
    cost_tests.append(report_backtest(b, f"slippage_{slippage_bp:g}bp_per_side"))

pd.DataFrame(cost_tests).set_index("Strategy")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

cols = ["net_equal_ret"]
if "net_ret_on_aum" in bt.columns:
    cols.append("net_ret_on_aum")

for col in cols:
    (1 + bt[col].fillna(0)).cumprod().plot(ax=ax, label=col)

ax.axhline(1, color="black", linewidth=0.8)
ax.set_title(label)
ax.set_ylabel("Cumulative return")
ax.legend()
ax.grid(alpha=0.25)
plt.show()


---
### **Minute Participation Event Backtest**

이 섹션은 뉴스별로 분봉 참여율 체결을 먼저 계산한 `participation_event_returns.parquet`를 사용합니다.

- 매수: 뉴스 발간 2분 후부터 당일 종가까지, 각 분 거래량의 5% 참여
- 매도: 익일 09:00부터 매수 수량이 소진될 때까지, 각 분 거래량의 5% 참여
- 동일 종목-동일일 복수 뉴스: threshold를 넘은 최초 뉴스만 포트폴리오에 사용
- 수익률: 매수 VWAP 대비 매도 VWAP, 매수/매도 수수료와 매도세 반영


In [ ]:
from intraday_experiment.participation_execution import (
    performance_stats as participation_performance_stats,
    threshold_participation_backtest,
)

participation_events = pd.read_parquet("participation_event_returns.parquet")

display(
    pd.Series({
        "Events": len(participation_events),
        "Buy VWAP Valid": participation_events["buy_vwap"].notna().sum(),
        "Sell VWAP Valid": participation_events["sell_vwap"].notna().sum(),
        "Fully Sold Next Day": participation_events["fully_sold_next_day"].sum(),
        "Median Buy Value (억원)": participation_events["buy_value"].median() / 100_000_000,
        "Median Net Ret": participation_events["net_participation_ret"].median(),
    })
)


In [ ]:
def run_participation_threshold_tests(
    event_returns,
    *,
    threshold_grid=None,
    start_date=START_DATE,
    universe_mask=universe_mask,
):
    if threshold_grid is None:
        threshold_grid = {
            "NB_sentiment": (0.7, 0.8, 0.9, 0.95),
            # SESTM score is tightly concentrated near 0.5, so use a separate absolute threshold grid.
            "SESTM_sentiment": (0.50001, 0.505, 0.51, 0.52, 0.54),
        }

    rows = []
    daily_results = {}
    for score_col, thresholds in threshold_grid.items():
        for threshold in thresholds:
            daily, summary = threshold_participation_backtest(
                event_returns,
                score_col=score_col,
                threshold=threshold,
                start_date=start_date,
                universe_mask=universe_mask,
                return_col="net_participation_ret",
            )
            key = f"{score_col.replace('_sentiment', '')}_th{threshold:g}"
            rows.append(pd.concat([pd.Series({"Strategy": key}), summary]))
            daily_results[key] = daily
    return pd.DataFrame(rows).set_index("Strategy"), daily_results


participation_summary, participation_daily = run_participation_threshold_tests(
    participation_events
)

participation_summary


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for key, daily in participation_daily.items():
    if key.startswith("NB_th0.9"):
        r = daily["value_weighted_ret"].dropna()
        if not r.empty:
            (1 + r).cumprod().plot(ax=ax, label=f"{key} VW")
    if key.startswith("SESTM_th0.52"):
        r = daily["value_weighted_ret"].dropna()
        if not r.empty:
            (1 + r).cumprod().plot(ax=ax, label=f"{key} VW")

ax.axhline(1, color="black", linewidth=0.8)
ax.set_title("Minute participation execution: threshold 0.9")
ax.set_ylabel("Cumulative return")
if ax.has_data():
    ax.legend()
ax.grid(alpha=0.25)
plt.show()
